# Synthetic Data Generator Pipeline
### Step 2 : Build Premelt Observations Stage

## Overview

This notebook supports the synthetic pipeline stage documented by this notebook. It is part of the Synthetic support portion of the capstone workflow and should be read as a notebook-first implementation step rather than a separate pipeline redesign.


## Objectives

- Document how this notebook supports the synthetic pipeline stage documented by this notebook.
- Preserve the existing notebook-first execution flow, configuration usage, logger behavior, ledger behavior, and artifact outputs.
- Make the notebook easier to review by separating setup, validation, processing, outputs, and handoff context.
- Clarify whether the notebook is a support, testing, streaming, or Bronze-handoff step in the synthetic path.


## Code Reference

A detailed code-cell reference for this notebook is maintained in the project documentation:`docs/technical_reference/notebook_code_references/synthetic_02_build_premilt_observations_stage_code_reference.md`


In [ ]:
import os

from utils.database.postgres import (
    get_engine_from_env, 
    read_sql_dataframe,
)

from utils.synthetic.pipeline.premelt_stage_writer import (
    build_observations_premelt_stage,
    validate_observations_premelt_stage,
)

from utils.core.env_helpers import (
    env_int, 
    env_str,
)


In [ ]:
SCHEMA = env_str("CAPSTONE_SCHEMA", "capstone")

DATASET_ID = env_str(
    "SYNTHETIC_DATASET_ID",
    "pump_synthetic_v1",
    aliases=("DATASET_ID",),
)

RUN_ID = env_str(
    "SYNTHETIC_RUN_ID",
    "synthetic_run_001",
    aliases=("RUN_ID",),
)

ASSET_ID = env_str(
    "SYNTHETIC_ASSET_ID",
    "pump_asset_001",
)

IF_EXISTS_FLAG = "replace"
RANDOM_SEED = env_int("SYNTHETIC_RANDOM_SEED", 42)
NUMBER_OF_SENSORS = env_int("SYNTHETIC_SENSOR_COUNT", 52)
CHUNK_SIZE = env_int("SYNTHETIC_PREMELT_SOURCE_ROW_CHUNK_SIZE", 10000)

PREMELT_SOURCE_TABLE_NAME = env_str(
    "SYNTHETIC_SOURCE_STREAM_TABLE",
    "synthetic_pump_stream",
)

PREMELT_DESTINATION_TABLE_NAME = env_str(
    "SYNTHETIC_PREMELT_OBSERVATIONS_TABLE",
    "synthetic_observations_premelt_stage",
)

print("Premelt config")
print("schema:", SCHEMA)
print("dataset id:", DATASET_ID)
print("run id:", RUN_ID)
print("asset id:", ASSET_ID)
print("source table:", PREMELT_SOURCE_TABLE_NAME)
print("target table:", PREMELT_DESTINATION_TABLE_NAME)

---

In [10]:

engine = get_engine_from_env()


---

In [11]:

premelt_table_name = build_observations_premelt_stage(
    engine=engine,
    schema=SCHEMA,
    source_table=PREMELT_SOURCE_TABLE_NAME,
    target_table=PREMELT_DESTINATION_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    asset_id=ASSET_ID,
    if_exists=IF_EXISTS_FLAG,
)

print("Built table:", premelt_table_name)


Built table: synthetic_observations_premelt_stage


---

In [12]:

validation_dataframe = validate_observations_premelt_stage(
    engine=engine,
    schema=SCHEMA,
    table_name=PREMELT_DESTINATION_TABLE_NAME,
)

display(validation_dataframe)

,row_count,min_observation_index,max_observation_index,distinct_generated_row_id_count,min_batch_id,max_batch_id
0,72000,1,72000,72000,1,1


---

In [ ]:
inspection_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        dataset_id,
        run_id,
        asset_id,
        generated_row_id,
        observation_index,
        batch_id,
        row_in_batch,
        global_cycle_id,
        stream_state,
        phase,
        meta_episode_id,
        is_telemetry_event,
        telemetry_event_type,
        producer_send_attempt
    FROM "{SCHEMA}"."{PREMELT_DESTINATION_TABLE_NAME}"
    ORDER BY observation_index
    LIMIT 25
    """,
)

display(inspection_dataframe)

,dataset_id,run_id,asset_id,generated_row_id,observation_index,batch_id,row_in_batch,global_cycle_id,stream_state,phase,meta_episode_id,is_telemetry_event,telemetry_event_type,producer_send_attempt
0,pump_synthetic_v1,premelt_run_001,pump_asset_001,premelt_run_001_obs_000000000001,1,1,0,1,normal,normal,0,False,None,1
1,pump_synthetic_v1,premelt_run_001,pump_asset_001,premelt_run_001_obs_000000000002,2,1,1,2,normal,normal,0,False,None,1
2,pump_synthetic_v1,premelt_run_001,pump_asset_001,premelt_run_001_obs_000000000003,3,1,2,3,normal,normal,0,False,None,1
3,pump_synthetic_v1,premelt_run_001,pump_asset_001,premelt_run_001_obs_000000000004,4,1,3,4,normal,normal,0,False,None,1
4,pump_synthetic_v1,premelt_run_001,pump_asset_001,premelt_run_001_obs_000000000005,5,1,4,5,normal,normal,0,False,None,1
5,pump_synthetic_v1,premelt_run_001,pump_asset_001,premelt_run_001_obs_000000000006,6,1,5,6,normal,normal,0,False,None,1
6,pump_synthetic_v1,premelt_run_001,pump_asset_001,premelt_run_001_obs_000000000007,7,1,6,7,normal,normal,0,False,None,1
7,pump_synthetic_v1,premelt_run_001,pump_asset_001,premelt_run_001_obs_000000000008,8,1,7,8,normal,normal,0,False,None,1
8,pump_synthetic_v1,premelt_run_001,pump_asset_001,premelt_run_001_obs_000000000009,9,1,8,9,normal,normal,0,False,None,1
9,pump_synthetic_v1,premelt_run_001,pump_asset_001,premelt_run_001_obs_000000000010,10,1,9,10,normal,normal,0,False,None,1


----

In [14]:
# Dispose Engine
engine.dispose()

## Summary

This notebook preserves the current analytical behavior while documenting the role of the Synthetic support step in the capstone workflow. The code cells above should continue to produce the same configured outputs, artifacts, logger messages, and ledger entries as before this documentation pass.


## Next Stage

The premelt observations feed timestamp assignment and downstream message shaping.
